🌐 **English** | [한국어](dx_com_02_intermediate.ko.ipynb) | [日本語](dx_com_02_intermediate.ja.ipynb) | [中文](dx_com_02_intermediate.zh.ipynb)

# DEEPX Tutorial 05-2 - DX-COM Intermediate

This notebook moves from a successful compile to a controlled and explainable compilation workflow.

You will improve calibration decisions, build real Model Zoo models with PPU Type 0 and Type 1, and apply TopK-first optimization to YOLO26 detection, segmentation, and pose models (n by default; s, m, l, and x are one variable away).

> Part 2 of 3 of this tutorial. The course map and the full list of parts are in the Beginner notebook.


## Learning objectives

By the end, you should be able to:

- select representative calibration data and reproduce model preprocessing,
- explain which detection operations run on the NPU PPU and which remain on the host CPU,
- choose PPU Type 0 or Type 1 from the detection-head architecture,
- validate PPU layer mappings against an ONNX graph,
- compile Model Zoo YOLOv7 and YOLOX-S models with hardware PPU,
- compare PPU and non-PPU Model Zoo DXNN runtime behavior,
- explain which part of a YOLO26 DXNN runs on the host CPU and what TopK-first optimization changes there,
- find the head tensors of a YOLO26 ONNX model automatically,
- apply TopK-first optimization to YOLO26 detection, segmentation, and pose models of any size (n, s, m, l, x),
- check that the optimized model gives the same results, and measure the CPU time it saves and the end-to-end throughput, and
- run the optimized DXNN in DX-APP without changing the application.

This notebook does not modify the SDK source tree. All generated files are stored under `<dx-tutorials>/notebooks/T05-DX-Compiler/workspace`, which git ignores.

## Prerequisites

- Tutorial 05-1 completed: `dxcom` in `venv-dx-compiler-local` and the sample calibration dataset.
- DX-RT installed with a DEEPX NPU for the `dxrun` comparisons in sections 4 and 6.
- For section 6.8: the DX-RT Python environment (`venv-dx-runtime`, Tutorial 01) and DX-APP (Tutorial 02). Without them only the DX-APP cells are skipped.
- Downloads: YOLOv7 and YOLOX ONNX and DXNN files from the Model Zoo (about 290 MB), and the YOLO26 detection, segmentation, and pose ONNX files for section 6 (about 33 MB for the default `n` size).
- Time: the YOLOv7 640x640 compile is the longest in this series; expect tens of minutes on a laptop. Section 6 compiles six YOLO26 models (baseline and TopK for detection, segmentation, and pose); for `n` each takes about 2 minutes on an x86_64 desktop.

The next cell locates the SDK, checks these requirements, and prints a status table. Anything marked `MISSING` comes with the step that provides it.


## 1. Initialize the tutorial workspace

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import json, os, re, shutil, sys

paths = setup_tutorial(needs=["dx_com", "calibration_dataset", "dx_rt"])
paths.require("dx_com", "calibration_dataset", "dx_rt")   # dx_rt provides dxparse and dxrun

DXCOM_PYTHON = DX_COMPILER_VENV / "bin" / "python"
DXPARSE_PATH = shutil.which("dxparse")
DXRUN_PATH = shutil.which("dxrun")

# Files created by this tutorial stay under notebooks/T05-DX-Compiler/workspace (ignored by git).
MODEL_DIR = WORKSPACE_DIR / "models"
CONFIG_DIR = WORKSPACE_DIR / "configs"
OUTPUT_DIR = WORKSPACE_DIR / "outputs"
for path in (WORKSPACE_DIR, MODEL_DIR, CONFIG_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Reuse the SDK sample images without copying them into this workspace.
CALIBRATION_SOURCE = DX_COM_DIR / "calibration_dataset"
CALIBRATION_DIR = WORKSPACE_DIR / "calibration_dataset"
if CALIBRATION_DIR.is_symlink() and not CALIBRATION_DIR.exists():
    CALIBRATION_DIR.unlink()
if not CALIBRATION_DIR.exists():
    CALIBRATION_DIR.symlink_to(CALIBRATION_SOURCE, target_is_directory=True)

os.chdir(WORKSPACE_DIR)
print()
print(f"DX-COM       : {DXCOM_PATH}")
print(f"Workspace    : {WORKSPACE_DIR}")
print(f"Calibration  : {CALIBRATION_DIR} -> {CALIBRATION_SOURCE}")


In [ ]:
try:
    import onnx
except ImportError:
    !uv pip install --python "{sys.executable}" --quiet onnx
    import onnx

print("ONNX:", onnx.__version__)


## 2. Calibration is part of the model definition

Quantization observes floating-point tensors produced by the calibration inputs. A convenient but unrelated image set can produce a DXNN file while degrading accuracy.

| Decision | Recommended practice |
|---|---|
| Samples | Cover real lighting, scale, background, and class distributions |
| Preprocessing | Match training and evaluation exactly |
| Count | Start with the model recipe; increase only after measuring stability |
| Method | Start with the published configuration, then compare methods using metrics |
| Validation | Compare ONNX and DXNN outputs on a held-out dataset |

The SDK `calibration_dataset` link keeps this tutorial runnable, but it is not a substitute for a task-specific dataset.

### 2.1 Preprocessing order

A typical detection pipeline is:

```text
image → letterbox/pad → BGR-to-RGB → divide by 255 → HWC-to-CHW → add batch axis
```

Do not copy this sequence blindly. Inspect the model exporter and training code. A different pad location, pad value, color order, or normalization changes the calibration tensor distribution.

## 3. PPU overview

The **Post-Processing Unit (PPU)** is hardware inside the DEEPX NPU. For supported YOLO detection heads, it reduces the number of raw candidates that the host CPU must handle.

The PPU performs two selected operations:

1. **confidence filtering** removes candidates below the compile-time threshold, and
2. **class prediction** selects the strongest class for each remaining candidate.

The PPU does **not** perform Non-Maximum Suppression (NMS). NMS and application-level interpretation still run on the host CPU. The PPU also does not replace image preprocessing or NPU inference.

### 3.1 Where PPU fits in the detection pipeline

```text
Without hardware PPU
┌──────────┐   ┌───────────────┐   ┌──────────────────────────────┐   ┌─────────┐
│ Input    │ → │ NPU inference │ → │ CPU filtering/class selection│ → │ CPU NMS │
└──────────┘   └───────────────┘   └──────────────────────────────┘   └─────────┘

With PPU Type 0 or Type 1
┌──────────┐   ┌───────────────┐   ┌────────────────────────────┐   ┌─────────┐
│ Input    │ → │ NPU inference │ → │ PPU filter/class prediction│ → │ CPU NMS │
└──────────┘   └───────────────┘   └────────────────────────────┘   └─────────┘
                                           hardware                    host
```

| Stage | Without PPU | With PPU Type 0/1 |
|---|---|---|
| Neural-network inference | NPU | NPU |
| Confidence filtering | Host CPU | PPU hardware |
| Best-class selection | Host CPU | PPU hardware |
| Bounding-box decode | Model/type dependent | Model/type dependent |
| NMS | Host CPU | Host CPU |

The main benefit is lower host-CPU work and less intermediate detection data. The exact end-to-end gain depends on the model, threshold, scene, host CPU, and application post-processing.

### 3.2 Choose the mode from the model architecture

| Mode | Architecture | Typical models | Layer mapping | Execution |
|---|---|---|---|---|
| PPU Type 0 | Anchor-based | YOLOv3, YOLOv4, YOLOv5, YOLOv7 | Detection Conv node → number of anchors | PPU hardware |
| PPU Type 1 | Anchor-free | YOLOX, YOLOv8–YOLOv12 | `bbox`, `obj_conf` when present, and `cls_conf` nodes | PPU hardware |
| `pre_optimize()` | TopK-first ONNX rewrite | YOLOv8, v9, v11, v12, v13 and YOLO26 (detection, segmentation, pose); RTMDet | Per-scale head output tensors | NPU + host CPU graph |

Do not select a type from the model name alone. Confirm the exported ONNX head structure and use node names from that exact file.

### 3.3 Read a PPU configuration

```text
PPU configuration
├── type           selects the supported head architecture
├── conf_thres     fixed confidence threshold compiled into the DXNN
├── num_classes    class count of this exported model
├── activation     Type 0 activation, usually Sigmoid
└── layer          exact ONNX head-node mapping
```

| Field | Type 0 | Type 1 | Why it matters |
|---|:---:|:---:|---|
| `type` | Required | Required | Selects the hardware data path |
| `conf_thres` | Required | Required | Controls how many candidates leave the PPU |
| `num_classes` | Required | Required | Must match head-channel layout |
| `activation` | Required | Not used | Applies the anchor-based score activation |
| `layer` | Dictionary | List | Connects PPU inputs to exact ONNX nodes |
| `num_anchors` | Per layer | Not used | Must match the anchor count at each scale |
| `obj_conf` | Not used | Model dependent | YOLOX has a separate objectness branch |

**Important:** `conf_thres` is fixed during compilation. Changing it later requires a new DXNN. A higher value reduces host work but can also remove valid detections. Measure accuracy before deployment.

## 4. PPU Type 0 lab: YOLOv7 from the Model Zoo

YOLOv7 uses an anchor-based detection head, so this lab uses PPU Type 0. You will download the actual Model Zoo ONNX and its matching PPU JSON, verify the mapped Conv nodes, adapt the dataset path, and compile a DXNN.

### 4.1 Download the Model Zoo files safely

The next cell imports `download_file` from `tutorial_paths.py` (the shared helper used by Tutorials 05-2 and 05-3; open the file to read it). If a non-empty destination file already exists, the helper skips the download without making a network request. Delete that local file first when you need to download a fresh copy. For a new download, the helper checks the remote file size, writes to a temporary `.part` file, and replaces the destination only after a complete transfer. A DNS or network failure raises an exception and does not leave an empty final file.

This lab downloads the ONNX and PPU JSON for compilation. It also downloads the published non-PPU DXNN as the benchmark reference.

In [ ]:
from tutorial_paths import download_file


def tensor_shape(value_info):
    return [dim.dim_value or dim.dim_param for dim in value_info.type.tensor_type.shape.dim]

In [ ]:
YOLOV7_ONNX_URL = "https://sdk.deepx.ai/modelzoo/onnx/yolov7_640x640.onnx"
YOLOV7_PPU_JSON_URL = "https://sdk.deepx.ai/modelzoo/q-lite-json/2_5_0/yolov7_640x640_ppu.json"
YOLOV7_NON_PPU_DXNN_URL = "https://sdk.deepx.ai/modelzoo/q-lite-dxnn/2_5_0/yolov7_640x640.dxnn"
YOLOV7_ONNX = MODEL_DIR / "yolov7_640x640.onnx"
YOLOV7_PPU_JSON = CONFIG_DIR / "yolov7_640x640_ppu.modelzoo.json"
YOLOV7_NON_PPU_DXNN = MODEL_DIR / "yolov7_640x640_non_ppu.dxnn"

download_file(YOLOV7_ONNX_URL, YOLOV7_ONNX)
download_file(YOLOV7_PPU_JSON_URL, YOLOV7_PPU_JSON)
download_file(YOLOV7_NON_PPU_DXNN_URL, YOLOV7_NON_PPU_DXNN)

### 4.2 Inspect the ONNX and Type 0 mapping

For Type 0, `layer` is a dictionary. Each key must be the name of a detection-head Conv node, and `num_anchors` must match that scale. The output channel count follows:

```text
channels = num_anchors × (5 + num_classes)
         = 3 × (5 + 80)
         = 255
```

In [ ]:
yolov7_model = onnx.load(YOLOV7_ONNX)
onnx.checker.check_model(yolov7_model)
yolov7_config = json.loads(YOLOV7_PPU_JSON.read_text())

print("IR/opset:", yolov7_model.ir_version, [(item.domain or "ai.onnx", item.version) for item in yolov7_model.opset_import])
print("Inputs:", [(value.name, tensor_shape(value)) for value in yolov7_model.graph.input])
print("Outputs:", [(value.name, tensor_shape(value)) for value in yolov7_model.graph.output])
print(json.dumps(yolov7_config["ppu"], indent=2))

node_names = {node.name for node in yolov7_model.graph.node}
type0_layers = yolov7_config["ppu"]["layer"]
missing_type0_nodes = sorted(set(type0_layers) - node_names)
if missing_type0_nodes:
    raise ValueError(f"Type 0 nodes were not found in this ONNX: {missing_type0_nodes}")
print("Verified Type 0 nodes:", list(type0_layers))

The highlighted Conv nodes are the three scale-specific detection heads used by the Model Zoo PPU configuration.

![YOLOv7 Type 0 PPU head mapping](assets/yolov7-class-n80-ppu.png)

### 4.3 Adapt only the environment-specific path

The Model Zoo JSON contains the preprocessing recipe used for this model. Keep it unchanged for the lab and replace only the unavailable calibration-dataset path. For a product model, use a representative dataset from the deployment domain.

In [ ]:
yolov7_local_config = json.loads(YOLOV7_PPU_JSON.read_text())
yolov7_local_config["default_loader"]["dataset_path"] = "./calibration_dataset"
YOLOV7_LOCAL_JSON = CONFIG_DIR / "yolov7_640x640_ppu.local.json"
YOLOV7_LOCAL_JSON.write_text(json.dumps(yolov7_local_config, indent=2) + "\n")

print("Local config:", YOLOV7_LOCAL_JSON)
print("Dataset     :", yolov7_local_config["default_loader"]["dataset_path"])
print("PPU type    :", yolov7_local_config["ppu"]["type"])

### 4.4 Compile and inspect the Type 0 DXNN

The next code cell activates the DX-COM environment and runs `dxcom` directly. It is equivalent to entering the following commands in a separate terminal:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/notebooks/T05-DX-Compiler/workspace
dxcom -m models/yolov7_640x640.onnx \
      -c configs/yolov7_640x640_ppu.local.json \
      -o outputs/yolov7_type0_ppu \
      --gen_log \
      --export_html
```

The actual SDK and tutorial paths may differ. The code cell uses the paths loaded from `config.json`. If the output directory already contains a DXNN file, it prints a skip message and does not run `dxcom` again.

> **About the `2>&1 | sed ... | grep ...` tail in the code cell:** `dxcom` draws progress bars with cursor-movement escape codes that Jupyter cannot render, which otherwise shows up as hundreds of blank lines. The filter only drops those progress-bar lines; every `[INFO]`, `[WARNING]`, and `[ERROR]` message stays visible. In a terminal you can omit the filter. `--gen_log` keeps the complete output in `compiler.log`. The same filter is used in every compile cell of this notebook.


In [ ]:
YOLOV7_PPU_OUTPUT = OUTPUT_DIR / "yolov7_type0_ppu"

yolov7_dxnn_files = list(YOLOV7_PPU_OUTPUT.glob("*.dxnn"))
if yolov7_dxnn_files:
    print(f"Skip compilation: found {yolov7_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{YOLOV7_ONNX}" \
            -c "{YOLOV7_LOCAL_JSON}" \
            -o "{YOLOV7_PPU_OUTPUT}" \
            --gen_log \
            --export_html \
            2>&1 | sed -u 's/\x1b\[A//g; s/\r/\n/g' | grep --line-buffered -vP '^\s*$|\d+(\.\d+)?/\d+(\.\d+)? \['

yolov7_dxnn_files = list(YOLOV7_PPU_OUTPUT.glob("*.dxnn"))
if not yolov7_dxnn_files:
    raise FileNotFoundError("Type 0 compilation did not produce a DXNN file.")
YOLOV7_PPU_DXNN = max(yolov7_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("DXNN:", YOLOV7_PPU_DXNN)

!dxparse -m "{YOLOV7_PPU_DXNN}" -v

### 4.5 Compare PPU and non-PPU with `dxrun`

The non-PPU DXNN is downloaded directly from the Model Zoo, while the PPU DXNN is the result compiled in Section 4.4. The next cell runs both models for five seconds with synthetic input:

```bash
dxrun -m models/yolov7_640x640_non_ppu.dxnn --use-ort -t 5
dxrun -m outputs/yolov7_type0_ppu/<compiled-model>.dxnn --use-ort -t 5
```

The non-PPU model contains NPU and CPU tasks and returns raw detection values. The PPU model returns hardware-generated `BBOX` data and reduces the output handled by the host. PPU does **not** guarantee higher synthetic-input FPS: model scheduling, output transfer, and device state can make the non-PPU result faster in this isolated test. For a product decision, also measure full-pipeline latency and host CPU usage with real input.

In [ ]:
from IPython.display import Markdown, display

def benchmark_model(label, model_path, seconds=5):
    print(f"\n===== {label} =====")
    # `lines = !command` runs the shell command and captures its output lines.
    lines = !dxrun -m "{model_path}" --use-ort -t {seconds}
    print("\n".join(lines))
    match = re.search(r"FPS\s*:\s*([0-9.]+)", "\n".join(lines))
    if not match:
        raise RuntimeError(f"Could not read FPS from dxrun output for {label}.")
    return float(match.group(1))

def compare_ppu_runtime(non_ppu_model, ppu_model, ppu_label):
    non_ppu_fps = benchmark_model("Non-PPU Model Zoo DXNN", non_ppu_model)
    ppu_fps = benchmark_model(ppu_label, ppu_model)
    ratio = ppu_fps / non_ppu_fps
    table = (
        "| Variant | FPS | Relative to non-PPU |\n"
        "|---|---:|---:|\n"
        f"| Non-PPU Model Zoo DXNN | {non_ppu_fps:.2f} | 1.00x |\n"
        f"| {ppu_label} | {ppu_fps:.2f} | {ratio:.2f}x |"
    )
    display(Markdown(table))
    return {"non_ppu_fps": non_ppu_fps, "ppu_fps": ppu_fps, "ratio": ratio}

yolov7_ppu_comparison = compare_ppu_runtime(
    YOLOV7_NON_PPU_DXNN,
    YOLOV7_PPU_DXNN,
    "Type 0 PPU DXNN",
)

## 5. PPU Type 1 lab: YOLOX-S from the Model Zoo

YOLOX uses a decoupled anchor-free detection head. Each scale has separate bounding-box, objectness, and class-confidence branches, so this lab uses PPU Type 1.

### 5.1 Download the ONNX and Type 1 JSON

The same safe downloader is reused. The ONNX and PPU JSON are used for compilation, while the published non-PPU DXNN is used as the benchmark reference.

In [ ]:
YOLOX_ONNX_URL = "https://sdk.deepx.ai/modelzoo/onnx/yolox-s_640x640.onnx"
YOLOX_PPU_JSON_URL = "https://sdk.deepx.ai/modelzoo/q-lite-json/2_5_0/yolox-s_640x640_ppu.json"
YOLOX_NON_PPU_DXNN_URL = "https://sdk.deepx.ai/modelzoo/q-lite-dxnn/2_5_0/yolox-s_640x640.dxnn"
YOLOX_ONNX = MODEL_DIR / "yolox-s_640x640.onnx"
YOLOX_PPU_JSON = CONFIG_DIR / "yolox-s_640x640_ppu.modelzoo.json"
YOLOX_NON_PPU_DXNN = MODEL_DIR / "yolox-s_640x640_non_ppu.dxnn"

download_file(YOLOX_ONNX_URL, YOLOX_ONNX)
download_file(YOLOX_PPU_JSON_URL, YOLOX_PPU_JSON)
download_file(YOLOX_NON_PPU_DXNN_URL, YOLOX_NON_PPU_DXNN)

### 5.2 Inspect the ONNX and Type 1 mapping

For YOLOX, every scale maps three named nodes:

```text
feature map ─┬─ bbox branch ─────→ bbox
             ├─ object branch ───→ obj_conf
             └─ class branch ────→ cls_conf
```

The three entries correspond to the 80×80, 40×40, and 20×20 detection scales.

In [ ]:
yolox_model = onnx.load(YOLOX_ONNX)
onnx.checker.check_model(yolox_model)
yolox_config = json.loads(YOLOX_PPU_JSON.read_text())

print("IR/opset:", yolox_model.ir_version, [(item.domain or "ai.onnx", item.version) for item in yolox_model.opset_import])
print("Inputs:", [(value.name, tensor_shape(value)) for value in yolox_model.graph.input])
print("Outputs:", [(value.name, tensor_shape(value)) for value in yolox_model.graph.output])
print(json.dumps(yolox_config["ppu"], indent=2))

node_names = {node.name for node in yolox_model.graph.node}
type1_layers = yolox_config["ppu"]["layer"]
mapped_type1_nodes = {name for layer in type1_layers for name in layer.values()}
missing_type1_nodes = sorted(mapped_type1_nodes - node_names)
if missing_type1_nodes:
    raise ValueError(f"Type 1 nodes were not found in this ONNX: {missing_type1_nodes}")
print("Verified Type 1 nodes:", sorted(mapped_type1_nodes))

The colors show the `bbox`, `obj_conf`, and `cls_conf` branches that must be paired at each scale.

![YOLOX Type 1 PPU head mapping](assets/yolox-class-n80-ppu.png)

### 5.3 Adapt the calibration-dataset path

Keep the downloaded Type 1 mapping and YOLOX preprocessing. Replace only the Model Zoo build-machine dataset path for this exercise.

In [ ]:
yolox_local_config = json.loads(YOLOX_PPU_JSON.read_text())
yolox_local_config["default_loader"]["dataset_path"] = "./calibration_dataset"
YOLOX_LOCAL_JSON = CONFIG_DIR / "yolox-s_640x640_ppu.local.json"
YOLOX_LOCAL_JSON.write_text(json.dumps(yolox_local_config, indent=2) + "\n")

print("Local config:", YOLOX_LOCAL_JSON)
print("Dataset     :", yolox_local_config["default_loader"]["dataset_path"])
print("PPU type    :", yolox_local_config["ppu"]["type"])

### 5.4 Compile and inspect the Type 1 DXNN

The next code cell is equivalent to entering the following commands in a separate terminal:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/notebooks/T05-DX-Compiler/workspace
dxcom -m models/yolox-s_640x640.onnx \
      -c configs/yolox-s_640x640_ppu.local.json \
      -o outputs/yolox_s_type1_ppu \
      --gen_log \
      --export_html
```

The code cell resolves the actual paths automatically. If the output directory already contains a DXNN file, it skips `dxcom`. Compare its `dxparse -v` output with the Type 0 result, especially the output tensor layout and PPU metadata.

In [ ]:
YOLOX_PPU_OUTPUT = OUTPUT_DIR / "yolox_s_type1_ppu"

yolox_dxnn_files = list(YOLOX_PPU_OUTPUT.glob("*.dxnn"))
if yolox_dxnn_files:
    print(f"Skip compilation: found {yolox_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{YOLOX_ONNX}" \
            -c "{YOLOX_LOCAL_JSON}" \
            -o "{YOLOX_PPU_OUTPUT}" \
            --gen_log \
            --export_html \
            2>&1 | sed -u 's/\x1b\[A//g; s/\r/\n/g' | grep --line-buffered -vP '^\s*$|\d+(\.\d+)?/\d+(\.\d+)? \['

yolox_dxnn_files = list(YOLOX_PPU_OUTPUT.glob("*.dxnn"))
if not yolox_dxnn_files:
    raise FileNotFoundError("Type 1 compilation did not produce a DXNN file.")
YOLOX_PPU_DXNN = max(yolox_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("DXNN:", YOLOX_PPU_DXNN)

!dxparse -m "{YOLOX_PPU_DXNN}" -v

### 5.5 Compare PPU and non-PPU with `dxrun`

Repeat the same controlled test for YOLOX-S. The non-PPU reference is the published Model Zoo DXNN downloaded in Section 5.1; it is not compiled in this tutorial.

```bash
dxrun -m models/yolox-s_640x640_non_ppu.dxnn --use-ort -t 5
dxrun -m outputs/yolox_s_type1_ppu/<compiled-model>.dxnn --use-ort -t 5
```

Interpret this synthetic benchmark together with the task structure shown by `dxparse`. A PPU model can reduce host-side processing and output traffic even when this single FPS number does not increase.

In [ ]:
yolox_ppu_comparison = compare_ppu_runtime(
    YOLOX_NON_PPU_DXNN,
    YOLOX_PPU_DXNN,
    "Type 1 PPU DXNN",
)

## 6. YOLO26 TopK-first optimization (n / s / m / l / x)

PPU Type 0 and Type 1 move detection post-processing into NPU hardware. YOLO26 takes another route: its post-processing stays on the host CPU, and DX-COM's `pre_optimize()` rewrites that CPU part so it does much less work. This section shows where the work runs, what the rewrite changes, how to confirm that the results stay the same, and how much time it saves. Then it applies the same steps to segmentation and pose models.

### Where YOLO26 post-processing runs

A compiled YOLO26 DXNN holds two tasks. You will see them with `dxparse` in 6.6.

```text
image ─► npu_0 [NPU] ──────────► head tensors ─► cpu_0 [CPU, ONNX Runtime on the host] ─► [1, 300, 6]
         backbone, neck, and     box and class    Sigmoid on the class logits, box decoding,
         the head Convs          logits, 3 scales  keep the best 300 candidates
```

The NPU task ends at the last Conv of each head branch. Everything after that is ordinary ONNX, which DX-RT runs on the host CPU with ONNX Runtime; this is why `dxrun` needs `--use-ort`. At 640x640 the head produces 8,400 candidates, so the baseline CPU task applies Sigmoid to 8,400 x 80 = 672,000 class logits in every frame before it keeps 300 of them.

### What `pre_optimize()` changes

```text
Baseline : 8,400 candidates → Sigmoid on every class logit → decode → TopK 300
Optimized: 8,400 candidates → TopK 300 on the raw logits    → Sigmoid and decode for those 300 only
```

Sigmoid keeps the order of the scores, so a TopK on raw logits selects the same candidates. The output stays `[1, 300, 6]` (box `x1 y1 x2 y2`, score, class id), and section 6.5 shows that the values are identical. Only the amount of host-CPU work changes.

The YOLO26 pass is `yolo_no_dfl_postprocess` (YOLO26 boxes do not use DFL). DX-COM also has `yolo_dfl_postprocess` for YOLOv8 to v13 and `rtmdet_postprocess` for RTMDet; the DX-COM User Manual, *Pre-Optimize API*, describes all three.

### When it pays off

The saving is on the host CPU. It matters when the CPU part is a large share of each frame: on slower host CPUs such as Arm boards, and for models with many classes. On a fast x86_64 desktop, or for a model with only a few classes, the gain is small. Section 6.7 measures the CPU part and the end-to-end throughput separately so you can judge your own host.

### Two ways to apply the pass

| Method | How | Use it when |
|---|---|---|
| Python API | `dx_com.pre_optimize(model, passes=...)` writes a new ONNX file, which you then compile | you want to inspect and check the rewritten ONNX before compiling (this section) |
| Compile config | a `pre_optimize` block in the compile JSON; `dxcom` applies the pass while it compiles | the pass settings are checked and you want a single compile step (see the note in 6.3) |

### Section map

| Step | What you do | Result |
|---|---|---|
| 6.1 | Download the Model Zoo YOLO26 detection model | baseline ONNX and JSON |
| 6.2 | Find the head tensors automatically | the pass settings |
| 6.3 | Apply the pass | TopK ONNX |
| 6.4 | Compare the CPU part before and after | op and shape table |
| 6.5 | Check that the results are identical | ONNX Runtime comparison on a real image |
| 6.6 | Compile both models | two DXNN files and their NPU/CPU split |
| 6.7 | Measure | CPU-part time and `dxrun` FPS |
| 6.8 | Run the TopK DXNN in DX-APP | result images |
| 6.9 | Repeat 6.1 to 6.8 for segmentation and pose | |

`YOLO26_VARIANT` in 6.1 selects the Model Zoo size, and every file name, compile, and benchmark below follows it. Start with `n`, then change the variable and re-run the section for a larger model. Each size uses its own files under `models/`, `configs/`, and `outputs/`, so switching back to a compiled size skips the compile.

| `YOLO26_VARIANT` | Detection ONNX download | Compile time per model (x86_64 desktop) |
|---|---:|---|
| `n` (default) | 10 MB | about 2 minutes (measured) |
| `s` | 37 MB | about 2 minutes (measured) |
| `m` | 78 MB | not measured; expect several minutes |
| `l` | 95 MB | not measured; expect 10 minutes or more |
| `x` | 213 MB | not measured; expect 10 minutes or more |

### 6.1 Download and validate the baseline model

Set `YOLO26_VARIANT` and run the cell. The baseline ONNX and its matching Q-Lite JSON come from the Model Zoo (`yolo26-<variant>_640x640.onnx` and `.json`). The safe downloader prevents the empty-file error that can occur when a shell download fails but the next notebook cell continues.

In [ ]:
# Choose the YOLO26 size: "n", "s", "m", "l", or "x". Everything in section 6 follows this variable.
YOLO26_VARIANT = "n"

assert YOLO26_VARIANT in ("n", "s", "m", "l", "x"), f"Unknown YOLO26 variant: {YOLO26_VARIANT!r}"
YOLO26_NAME = f"yolo26-{YOLO26_VARIANT}_640x640"

YOLO26_ONNX_URL = f"https://sdk.deepx.ai/modelzoo/onnx/{YOLO26_NAME}.onnx"
YOLO26_JSON_URL = f"https://sdk.deepx.ai/modelzoo/q-lite-json/2_5_0/{YOLO26_NAME}.json"
YOLO26_BASELINE = MODEL_DIR / f"{YOLO26_NAME}.onnx"
YOLO26_TOPK = MODEL_DIR / f"{YOLO26_NAME}_topk300.onnx"
YOLO26_MODELZOO_JSON = CONFIG_DIR / f"{YOLO26_NAME}.modelzoo.json"
YOLO26_LOCAL_JSON = CONFIG_DIR / f"{YOLO26_NAME}.local.json"
YOLO26_BASELINE_OUTPUT = OUTPUT_DIR / f"yolo26{YOLO26_VARIANT}_baseline"
YOLO26_TOPK_OUTPUT = OUTPUT_DIR / f"yolo26{YOLO26_VARIANT}_topk300"

print(f"YOLO26 variant : {YOLO26_VARIANT}  ({YOLO26_NAME})")
download_file(YOLO26_ONNX_URL, YOLO26_BASELINE)
download_file(YOLO26_JSON_URL, YOLO26_MODELZOO_JSON)


In [ ]:
yolo26 = onnx.load(YOLO26_BASELINE)
onnx.checker.check_model(yolo26)

print("IR/opset:", yolo26.ir_version, [(item.domain or "ai.onnx", item.version) for item in yolo26.opset_import])
print("Inputs:", [(value.name, tensor_shape(value)) for value in yolo26.graph.input])
print("Outputs:", [(value.name, tensor_shape(value)) for value in yolo26.graph.output])
print("Nodes before optimization:", len(yolo26.graph.node))

### 6.2 Find the head tensors

`pre_optimize()` needs to know where the NPU part ends: for each detection scale, the output tensor of the last Conv in each head branch. These are tensor names from your exact ONNX file. They depend on the model and the exporter version, so never copy them from another model.

A head tensor is easy to recognize:

1. It is the output of a `Conv` with shape `[1, C, H, W]`, where `H = 640 / stride` and the stride is 8, 16, or 32 (80x80, 40x40, 20x20).
2. No activation follows it; only `Reshape` or `Concat` use it.
3. Its channel count `C` gives its role:

| Role | Channels | Tasks |
|---|---|---|
| `bbox` | 4 (distances to the four box sides) | all |
| `cls_conf` | number of classes (class logits) | all |
| `mask_coeff` | 32 | `seg` |
| `kpt` | keypoints x values per keypoint (17 x 3 = 51) | `pose` |

Real models add two details:

- **Equal channel counts.** In a 4-class model `cls_conf` has 4 channels, like `bbox`. Only the class-score branch goes straight into a `Sigmoid`, which tells the two apart.
- **Two heads.** YOLO26 is trained with a one-to-many and a one-to-one head. Inference uses the one-to-one head, `one2one_*` in the tensor names. An export that contains only the one-to-many head does not work with this pass (see the note on `nms=False` in 6.3).

The next cell writes `yolo26_heads.py`, a small module that applies these rules. It reads the task, class count, and keypoint layout from the Ultralytics metadata in the ONNX file, and it stops with an explanation when a rule does not hold. It needs only the `onnx` package, so the notebook and the DX-COM environment can both import it. The DX-COM User Manual, *Identifying Conv Head Tensor Names*, explains the same rules and how to check them in Netron.

In [ ]:
%%writefile yolo26_heads.py
"""Find the YOLO26 head tensors for dx_com.pre_optimize and extract the host-CPU part of a model.

Needs only the onnx package, so the notebook kernel and the DX-COM environment can both import it.
"""
import ast
from collections import defaultdict

import onnx
from onnx import shape_inference

TASKS = {"detect": "base", "segment": "seg", "pose": "pose"}  # Ultralytics metadata -> pre_optimize task
MASK_CHANNELS = 32
RESHAPING_OPS = {"Reshape", "Concat", "Transpose", "Flatten", "Squeeze", "Unsqueeze"}


def read_metadata(model):
    """Task, class names, keypoint layout, and end-to-end flag from the Ultralytics ONNX metadata."""
    meta = {prop.key: prop.value for prop in model.metadata_props}
    names = ast.literal_eval(meta["names"]) if "names" in meta else None
    return {
        "task": TASKS.get(meta.get("task")),
        "names": names,
        "num_classes": len(names) if names else None,
        "kpt_shape": list(ast.literal_eval(meta["kpt_shape"])) if "kpt_shape" in meta else None,
        "end2end": meta.get("end2end"),
    }


def tensor_shapes(model):
    graph = shape_inference.infer_shapes(model).graph
    return {
        value.name: [dim.dim_value for dim in value.type.tensor_type.shape.dim]
        for value in list(graph.input) + list(graph.value_info) + list(graph.output)
    }


def find_yolo26_heads(model, task=None, num_classes=None, kpt_shape=None):
    """Return the yolo_no_dfl_postprocess settings (task, layers, num_classes) for a YOLO26 ONNX model.

    A head tensor is the output of the last Conv of a head branch:
      1. it is 4D, [1, C, H, W], at stride 8, 16, or 32 of the input,
      2. only Reshape / Concat / Transpose consume it (no activation follows), and
      3. its channel count C gives its role: bbox 4, cls_conf num_classes,
         mask_coeff 32 (seg), kpt keypoints x values per keypoint (pose).
    When two roles have the same channel count (a 4-class model), the class-score head is the
    one that goes straight into a Sigmoid. Values left as None come from the Ultralytics metadata.
    """
    meta = read_metadata(model)
    if meta["end2end"] == "False":
        raise ValueError(
            "This ONNX file has the one-to-many head (output [1, 4 + classes, 8400]), not the NMS-free head "
            "that yolo_no_dfl_postprocess expects. Re-export it with: yolo export ... format=onnx nms=False")
    task = task or meta["task"]
    num_classes = num_classes or meta["num_classes"]
    kpt_shape = kpt_shape or meta["kpt_shape"]
    if task not in ("base", "seg", "pose"):
        raise ValueError(f"Unknown task {task!r}: use 'base', 'seg', or 'pose'.")
    if meta["task"] and meta["task"] != task:
        raise ValueError(f"The model metadata says task={meta['task']!r}, not {task!r}. "
                         "With the wrong task, pre_optimize can silently drop the mask or keypoint outputs.")
    if num_classes is None:
        raise ValueError("No class names in the ONNX metadata: pass num_classes.")
    if task == "pose" and kpt_shape is None:
        raise ValueError("No kpt_shape in the ONNX metadata: pass kpt_shape, for example [17, 3].")

    roles = {"bbox": 4, "cls_conf": num_classes}
    if task == "seg":
        roles["mask_coeff"] = MASK_CHANNELS
    elif task == "pose":
        roles["kpt"] = kpt_shape[0] * kpt_shape[1]

    graph = model.graph
    shapes = tensor_shapes(model)
    consumers = defaultdict(list)
    for node in graph.node:
        for name in node.input:
            consumers[name].append(node)

    def first_ops(tensor):
        """Op types of the first nodes after a tensor that do more than reshape it."""
        found, pending = set(), [tensor]
        while pending:
            for node in consumers[pending.pop()]:
                if node.op_type in RESHAPING_OPS:
                    pending.extend(node.output)
                else:
                    found.add(node.op_type)
        return found

    input_h = shapes[graph.input[0].name][2]
    if not input_h:
        raise ValueError("The input height is dynamic: re-export the model with a fixed imgsz.")

    heads = defaultdict(lambda: defaultdict(list))  # stride -> channels -> [tensor names]
    for node in graph.node:
        if node.op_type != "Conv":
            continue
        out = node.output[0]
        shape = shapes.get(out)
        if not shape or len(shape) != 4 or not shape[2] or input_h % shape[2]:
            continue
        stride = input_h // shape[2]
        if stride in (8, 16, 32) and {c.op_type for c in consumers[out]} <= RESHAPING_OPS:
            heads[stride][shape[1]].append(out)

    layers = []
    for stride in (8, 16, 32):  # pre_optimize needs the layers in stride order
        layer = {}
        for role, channels in roles.items():
            names = heads[stride].get(channels, [])
            names = [name for name in names if "one2one" in name] or names  # inference head of YOLO26
            names = [name for name in names if ("Sigmoid" in first_ops(name)) == (role == "cls_conf")]
            if len(names) != 1:
                raise ValueError(f"stride {stride}: expected one {role} head Conv with {channels} channels, "
                                 f"found {names or 'none'}. Check num_classes, or pick the tensor in Netron.")
            layer[role] = names[0]
        if task == "pose":
            layer["kpt_shape"] = list(kpt_shape)
        layers.append(layer)
    return {"task": task, "layers": layers, "num_classes": num_classes}


def head_tensors(config):
    """All head tensor names in a pass configuration."""
    return [name for layer in config["layers"] for role, name in layer.items() if role != "kpt_shape"]


def describe_heads(model, config):
    """One line per scale: the channel count of every head tensor."""
    shapes = tensor_shapes(model)
    lines = []
    for stride, layer in zip((8, 16, 32), config["layers"]):
        roles = {role: name for role, name in layer.items() if role != "kpt_shape"}
        _, _, height, width = shapes[roles["bbox"]]
        lines.append(f"stride {stride:2d} ({height}x{width}): "
                     + ", ".join(f"{role} {shapes[name][1]} ch" for role, name in roles.items()))
    return lines


def extract_cpu_tail(model_path, config, destination):
    """Save the part of the model after the head tensors: what the host CPU runs in the compiled DXNN."""
    output = onnx.load(model_path, load_external_data=False).graph.output[0].name
    onnx.utils.extract_model(str(model_path), str(destination), head_tensors(config), [output])
    return destination


def cpu_tail_ops(tail_path, op_types=("ReduceMax", "TopK", "Sigmoid")):
    """(op type, output shape) of the ops whose work grows with the number of candidates."""
    model = onnx.load(tail_path)
    shapes = tensor_shapes(model)
    return [(node.op_type, shapes.get(node.output[0])) for node in model.graph.node if node.op_type in op_types]

In [ ]:
import importlib

if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
import yolo26_heads; importlib.reload(yolo26_heads)  # pick up edits to the module without a restart
from yolo26_heads import (cpu_tail_ops, describe_heads, extract_cpu_tail, find_yolo26_heads,
                          head_tensors, read_metadata)

yolo26_pass_config = find_yolo26_heads(yolo26)
print(f"task={yolo26_pass_config['task']}, num_classes={yolo26_pass_config['num_classes']}")
print("\n".join(describe_heads(yolo26, yolo26_pass_config)))
print(json.dumps(yolo26_pass_config["layers"], indent=2))

### 6.3 Apply `yolo_no_dfl_postprocess`

`optimize_yolo26.py` runs in the DX-COM Python environment, where `dx_com` is installed. It finds the head tensors with `yolo26_heads.py`, prints the pass settings, and writes the TopK model to a separate ONNX file, so the baseline stays unchanged. The same script serves every size and task, and later your own model.

The pass settings are:

| Key | Value | Notes |
|---|---|---|
| `task` | `base`, `seg`, or `pose` | required; selects which head tensors each layer must have |
| `layers` | one entry per scale, ordered by stride 8, 16, 32 | from 6.2 |
| `num_classes` | class count | default 80 |
| `topk` | candidates kept before decoding | default 300; the Model Zoo pre-optimized profiles use 512 |

> **Changed in DX-COM v2.5.0.** The pass formerly called `yolo26_postprocess` is now `yolo_no_dfl_postprocess`, and `yolo_postprocess` (YOLOv8 to v13) is now `yolo_dfl_postprocess`. The old names raise an error. Every YOLO pass needs a `task` key, and `input_height` / `input_width` are no longer accepted (the input size comes from the model).

> **Your own YOLO26 model.** The script also works for a model you trained, if it was exported with the NMS-free head: `yolo export model=best.pt format=onnx imgsz=640 batch=1 nms=False`. Without `nms=False`, current Ultralytics releases (tested with 8.4.208) export the one-to-many head with the output `[1, 4 + classes, 8400]`, and `yolo26_heads.py` stops with a re-export message. Keep `imgsz` fixed; do not use `dynamic=True`. `num_classes` and, for pose, `kpt_shape` come from the metadata of your export.

> **Applying the pass from the compile JSON.** The script prints the settings as `{"yolo_no_dfl_postprocess": {...}}`. To let `dxcom` apply the pass while it compiles the original ONNX, put that dictionary in a list under `pre_optimize` in the compile JSON: `"pre_optimize": [{"yolo_no_dfl_postprocess": {...}}]`. The compile log then shows `[pre_optimize] applying pass 'yolo_no_dfl_postprocess' from config`, and the DXNN keeps the ONNX file name. Do not put `pre_optimize` and `ppu` in the same JSON: DX-COM 2.5.0 compiles with `ppu`, drops `pre_optimize`, and only prints `falling back to legacy: ValueError: config 'pre_optimize' and 'ppu' are mutually exclusive`.

The next two cells write the script and run it. For the default `n` variant this is equivalent to:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/notebooks/T05-DX-Compiler/workspace
python optimize_yolo26.py models/yolo26-n_640x640.onnx models/yolo26-n_640x640_topk300.onnx
```

In [ ]:
%%writefile optimize_yolo26.py
"""Apply the DX-COM yolo_no_dfl_postprocess (TopK-first) pass to a YOLO26 ONNX export.

Usage:
  python optimize_yolo26.py <baseline.onnx> <optimized.onnx>
                            [--task base|seg|pose] [--num-classes N] [--kpt-shape K V] [--topk 300]

yolo26_heads.py finds the head tensors and reads the task, class count, and keypoint layout
from the Ultralytics metadata. The options are only needed when the metadata is missing.
"""
import argparse
import json
from pathlib import Path

import onnx
import dx_com

from yolo26_heads import find_yolo26_heads

parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
parser.add_argument("source", type=Path)
parser.add_argument("destination", type=Path)
parser.add_argument("--task", choices=["base", "seg", "pose"])
parser.add_argument("--num-classes", type=int)
parser.add_argument("--kpt-shape", type=int, nargs=2, metavar=("KEYPOINTS", "VALUES"))
parser.add_argument("--topk", type=int, default=300)
args = parser.parse_args()

model = onnx.load(args.source)
config = find_yolo26_heads(model, task=args.task, num_classes=args.num_classes, kpt_shape=args.kpt_shape)
config["topk"] = args.topk
print(json.dumps({"yolo_no_dfl_postprocess": config}, indent=2))

optimized = dx_com.pre_optimize(model, passes={"yolo_no_dfl_postprocess": config})
onnx.checker.check_model(optimized)
onnx.save(optimized, args.destination)
print(f"{args.source.name} -> {args.destination}")

In [ ]:
TOPK_SCRIPT = WORKSPACE_DIR / "optimize_yolo26.py"

!"{DXCOM_PYTHON}" "{TOPK_SCRIPT}" "{YOLO26_BASELINE}" "{YOLO26_TOPK}"

In [ ]:
optimized_yolo26 = onnx.load(YOLO26_TOPK)
onnx.checker.check_model(optimized_yolo26)

print("Baseline outputs :", [(value.name, tensor_shape(value)) for value in yolo26.graph.output])
print("Optimized outputs:", [(value.name, tensor_shape(value)) for value in optimized_yolo26.graph.output])
print("Baseline nodes   :", len(yolo26.graph.node))
print("Optimized nodes  :", len(optimized_yolo26.graph.node))

### 6.4 Compare the CPU part before and after

The CPU task of the compiled DXNN is the ONNX subgraph from the head tensors to the model output. The next cell cuts that subgraph out of both models (`onnx.utils.extract_model`) and lists the ops whose work grows with the number of candidates.

What to look for:

- **Baseline:** `Sigmoid` runs on `[1, 80, 8400]`, one value per class for every candidate, and the TopK comes after it.
- **TopK model:** `ReduceMax` and `TopK` select 300 candidates from the raw logits first, and `Sigmoid` runs on `[1, 300, 80]`, which is 28 times fewer values.

The TopK model has more nodes, because it adds the gather steps that pick the 300 candidates. Those extra nodes work on 300 rows, not 8,400. The cut-out CPU parts are also used to time the CPU work in 6.7.

In [ ]:
import numpy as np
from IPython.display import Markdown, display


def compare_cpu_tails(baseline_path, topk_path, config, label):
    "Cut out the host-CPU part of both models and list the ops whose work grows with the candidate count."
    tails, rows = {}, []
    for name, path in (("baseline", baseline_path), ("TopK", topk_path)):
        tails[name] = extract_cpu_tail(path, config, MODEL_DIR / f"{Path(path).stem}.cpu_part.onnx")
        for op, shape in cpu_tail_ops(tails[name]):
            rows.append(f"| {name} | `{op}` | `{shape}` | {int(np.prod(shape)):,} |")
    display(Markdown(f"**{label}: CPU part**\n\n| Model | Op | Output shape | Values |\n|---|---|---|---:|\n"
                     + "\n".join(rows)))
    return tails


yolo26_tails = compare_cpu_tails(YOLO26_BASELINE, YOLO26_TOPK, yolo26_pass_config, f"YOLO26-{YOLO26_VARIANT}")

### 6.5 Check that the results are identical

A changed graph must give the same answers before it is worth compiling. The next cells run both ONNX models with ONNX Runtime on the same real image, compare the outputs, and draw the detections of each model.

The image is prepared the way the Model Zoo JSON prepares calibration images: padded to a 640x640 square with the value 114, converted from BGR to RGB, divided by 255, and changed to NCHW. The test image comes from the DX-APP samples; if DX-APP is not installed, the first calibration image is used.

The comparison covers every candidate with a score above 0.001, the threshold the DX Model Zoo uses for mAP evaluation, and every other output (for example the segmentation prototype). Candidates with lower scores are left out on purpose: far down the list many candidates have exactly the same score, and TopK may return tied candidates in either order. The rewritten model sorts raw logits instead of Sigmoid outputs, so the order of such ties can differ. On the test images of this notebook the first tie appeared below a score of 0.0001.

The comparison checks the rewritten ONNX graph. Compiling quantizes each model separately, so the two DXNN files can differ slightly; 6.8 runs both on the NPU.

In [ ]:
import cv2
import matplotlib.pyplot as plt

try:
    import onnxruntime as ort
except ImportError:
    !uv pip install --python "{sys.executable}" --quiet onnxruntime
    import onnxruntime as ort
ort.set_default_logger_severity(3)  # hide notices about unused initializers


def letterbox(image_path, size=640):
    "Model Zoo preprocessing: pad to a square with 114, BGR to RGB, divide by 255, NCHW."
    image = cv2.imread(str(image_path))
    height, width = image.shape[:2]
    scale = min(size / height, size / width)
    new_h, new_w = round(height * scale), round(width * scale)
    canvas = np.full((size, size, 3), 114, np.uint8)
    top, left = (size - new_h) // 2, (size - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = cv2.resize(image, (new_w, new_h))
    tensor = canvas[:, :, ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255
    return canvas, np.ascontiguousarray(tensor)


def run_onnx(model_path, tensor):
    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    return session.run(None, {session.get_inputs()[0].name: tensor})


def compare_onnx_outputs(baseline_path, topk_path, tensor, score_threshold=0.25, compare_threshold=0.001):
    "Run both models on one input; compare every candidate above compare_threshold and the other outputs."
    baseline, topk = run_onnx(baseline_path, tensor), run_onnx(topk_path, tensor)
    rows = [outputs[0][0][outputs[0][0][:, 4] > compare_threshold] for outputs in (baseline, topk)]
    if len(rows[0]) == len(rows[1]):
        difference = f"largest difference {np.abs(rows[0] - rows[1]).max(initial=0):.6g}"
    else:
        difference = "DIFFERENT number of candidates"
    print(f"output 0 {list(baseline[0].shape)}: candidates with score > {compare_threshold}: "
          f"baseline {len(rows[0])}, TopK {len(rows[1])}, {difference}")
    for index, (a, b) in enumerate(zip(baseline[1:], topk[1:]), start=1):
        print(f"output {index} {list(a.shape)}: largest difference {np.abs(a - b).max():.6g}")
    counts = [int((outputs[0][0][:, 4] > score_threshold).sum()) for outputs in (baseline, topk)]
    print(f"detections with score > {score_threshold}: baseline {counts[0]}, TopK {counts[1]}")
    return baseline, topk


def draw_results(canvas, outputs, names, task="base", kpt_shape=None, score_threshold=0.25):
    "Draw a [1, 300, 6 + extra] YOLO26 result (boxes, plus masks or keypoints) on the 640x640 input."
    image = canvas.copy()
    rows = outputs[0][0]
    for row in rows[rows[:, 4] > score_threshold]:
        x1, y1, x2, y2 = (int(round(v)) for v in row[:4])
        if task == "seg":  # mask = sigmoid(coefficients x prototypes), kept inside the box
            mask = 1 / (1 + np.exp(-np.tensordot(row[6:38], outputs[1][0], axes=1)))
            inside = np.zeros((640, 640), bool)
            inside[max(y1, 0):y2, max(x1, 0):x2] = True
            selected = (cv2.resize(mask, (640, 640)) > 0.5) & inside
            image[selected] = (0.5 * image[selected] + (0, 110, 0)).astype(np.uint8)
        if task == "pose":
            for x, y, *visible in row[6:].reshape(kpt_shape):
                if not visible or visible[0] > 0.5:
                    cv2.circle(image, (int(x), int(y)), 3, (0, 0, 255), -1)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 220, 0), 2)
        cv2.putText(image, f"{names[int(row[5])]} {row[4]:.2f}", (x1, max(y1 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 0), 1, cv2.LINE_AA)
    return image


def show_images(images, width=6):
    "Show BGR images side by side; images is {title: image}."
    figure, axes = plt.subplots(1, len(images), figsize=(width * len(images), width))
    for axis, (title, image) in zip(np.atleast_1d(axes), images.items()):
        axis.imshow(image[:, :, ::-1])
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


TEST_IMAGE = DX_APP_DIR / "sample" / "img" / "sample_people.jpg"
if not TEST_IMAGE.is_file():
    TEST_IMAGE = sorted(CALIBRATION_DIR.glob("*.jp*g"))[0]
test_canvas, test_tensor = letterbox(TEST_IMAGE)
print("Test image:", TEST_IMAGE)

In [ ]:
yolo26_outputs = compare_onnx_outputs(YOLO26_BASELINE, YOLO26_TOPK, test_tensor)
yolo26_names = read_metadata(yolo26)["names"]
show_images({
    f"YOLO26-{YOLO26_VARIANT} baseline ONNX": draw_results(test_canvas, yolo26_outputs[0], yolo26_names),
    f"YOLO26-{YOLO26_VARIANT} TopK ONNX": draw_results(test_canvas, yolo26_outputs[1], yolo26_names),
})

A largest difference of 0 with the same number of candidates means the rewrite gives the same result for this image. For your own model, run the same check on several images from your data, including images with many objects and images with none.

### 6.6 Compile both models

#### 6.6.1 Adapt the Model Zoo configuration

Both models use the same calibration and preprocessing configuration, so the ONNX graph is the only difference between the two compiles. Only the dataset path, which points to a Model Zoo server, is changed.

In [ ]:
yolo26_config = json.loads(YOLO26_MODELZOO_JSON.read_text())
yolo26_config["default_loader"]["dataset_path"] = "./calibration_dataset"
YOLO26_LOCAL_JSON.write_text(json.dumps(yolo26_config, indent=2) + "\n")
print(YOLO26_LOCAL_JSON.read_text())

#### 6.6.2 Compile the baseline

For the default `n` variant, the next code cell is equivalent to entering the following commands in a separate terminal (other variants change `yolo26-n` and `yolo26n` accordingly):

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/notebooks/T05-DX-Compiler/workspace
dxcom -m models/yolo26-n_640x640.onnx \
      -c configs/yolo26-n_640x640.local.json \
      -o outputs/yolo26n_baseline \
      --gen_log \
      --export_html
```

If the output directory already contains a DXNN file, the code cell reports that file and skips `dxcom`.

In [ ]:
baseline_dxnn_files = list(YOLO26_BASELINE_OUTPUT.glob("*.dxnn"))
if baseline_dxnn_files:
    print(f"Skip compilation: found {baseline_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{YOLO26_BASELINE}" \
            -c "{YOLO26_LOCAL_JSON}" \
            -o "{YOLO26_BASELINE_OUTPUT}" \
            --gen_log \
            --export_html \
            2>&1 | sed -u 's/\x1b\[A//g; s/\r/\n/g' | grep --line-buffered -vP '^\s*$|\d+(\.\d+)?/\d+(\.\d+)? \['

baseline_dxnn_files = list(YOLO26_BASELINE_OUTPUT.glob("*.dxnn"))
if not baseline_dxnn_files:
    raise FileNotFoundError("Baseline compilation did not produce a DXNN file.")
yolo26_baseline_dxnn = max(baseline_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("Baseline DXNN:", yolo26_baseline_dxnn)

#### 6.6.3 Compile the TopK model

For the default `n` variant, the next code cell is equivalent to entering the following commands in a separate terminal (other variants change `yolo26-n` and `yolo26n` accordingly):

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/notebooks/T05-DX-Compiler/workspace
dxcom -m models/yolo26-n_640x640_topk300.onnx \
      -c configs/yolo26-n_640x640.local.json \
      -o outputs/yolo26n_topk300 \
      --gen_log \
      --export_html
```

If the output directory already contains a DXNN file, the code cell reports that file and skips `dxcom`.

In [ ]:
topk_dxnn_files = list(YOLO26_TOPK_OUTPUT.glob("*.dxnn"))
if topk_dxnn_files:
    print(f"Skip compilation: found {topk_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{YOLO26_TOPK}" \
            -c "{YOLO26_LOCAL_JSON}" \
            -o "{YOLO26_TOPK_OUTPUT}" \
            --gen_log \
            --export_html \
            2>&1 | sed -u 's/\x1b\[A//g; s/\r/\n/g' | grep --line-buffered -vP '^\s*$|\d+(\.\d+)?/\d+(\.\d+)? \['

topk_dxnn_files = list(YOLO26_TOPK_OUTPUT.glob("*.dxnn"))
if not topk_dxnn_files:
    raise FileNotFoundError("TopK compilation did not produce a DXNN file.")
yolo26_topk_dxnn = max(topk_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("TopK DXNN:", yolo26_topk_dxnn)

#### 6.6.4 See the NPU/CPU split

`dxparse -v` lists the tasks of a DXNN and the tensors that pass between them. The next cell prints only that part. In the baseline, `npu_0` ends at the head tensors found in 6.2 and `cpu_0` turns them into `output0`. In the TopK model the NPU also changes the layout of the head tensors for the CPU part (`preopt_*_tr_*`, channels last), and `cpu_0` produces `preopt_output`.

In [ ]:
def show_task_split(dxnn_path):
    "Print each DXNN task with its input and output tensors, from dxparse -v."
    print(f"===== {Path(dxnn_path).name} =====")
    section = None
    for line in get_ipython().getoutput(f'dxparse -m "{dxnn_path}" -v'):
        task = re.match(r"Task\[\d+\]: (\S+ \[\w+\])", line)
        if task:
            print(task.group(1))
        elif re.search(r"\+- (Inputs|Outputs):", line):
            section = "in " if "Inputs" in line else "out"
        elif section and (tensor := re.search(r"\+- (\S+) \{shape: (\[[^\]]*\])", line)):
            print(f"  {section} {tensor.group(1)} {tensor.group(2)}")
        elif not line.strip(" |"):
            section = None


show_task_split(yolo26_baseline_dxnn)
show_task_split(yolo26_topk_dxnn)

### 6.7 Measure the effect

Two measurements answer two questions:

1. **How much CPU work does the pass remove?** The CPU parts from 6.4 run with ONNX Runtime on one thread, fed with the real head tensors of the test image. This isolates the work that the pass changes.
2. **Does the whole pipeline get faster?** `dxrun --use-ort -t 5` runs each DXNN for five seconds with synthetic input, NPU and CPU parts together.

DX-RT overlaps the NPU and CPU tasks of consecutive frames, so end-to-end FPS is set by the slower of the two. When the NPU is the slower stage, which is common on a fast x86_64 host, a shorter CPU part changes FPS very little. On a slower host CPU the CPU part becomes the limit, and the time saved in measurement 1 turns into FPS.

For the default `n` variant the `dxrun` commands are:

```bash
dxrun -m outputs/yolo26n_baseline/yolo26-n_640x640.dxnn --use-ort -t 5
dxrun -m outputs/yolo26n_topk300/yolo26-n_640x640_topk300.dxnn --use-ort -t 5
```

Run both models on the same device under similar load and temperature, and repeat the measurement before you draw conclusions. These numbers do not measure accuracy or the FPS of a full video application.

In [ ]:
import time


def head_values(baseline_path, config, tensor):
    "Run the baseline ONNX once and return the head tensors: the real input of the CPU part."
    model = onnx.load(baseline_path)
    names = head_tensors(config)
    model.graph.output.extend(onnx.helper.make_tensor_value_info(name, onnx.TensorProto.FLOAT, None)
                              for name in names)
    session = ort.InferenceSession(model.SerializeToString(), providers=["CPUExecutionProvider"])
    return dict(zip(names, session.run(names, {session.get_inputs()[0].name: tensor})))


def time_cpu_part(tail_path, feeds, runs=300):
    "Average time of one CPU-part run on a single thread, in milliseconds."
    options = ort.SessionOptions()
    options.intra_op_num_threads = options.inter_op_num_threads = 1
    session = ort.InferenceSession(str(tail_path), options, providers=["CPUExecutionProvider"])
    inputs = {item.name: feeds[item.name] for item in session.get_inputs()}
    for _ in range(20):
        session.run(None, inputs)
    start = time.perf_counter()
    for _ in range(runs):
        session.run(None, inputs)
    return (time.perf_counter() - start) / runs * 1000


def benchmark_dxnn(label, model_path, seconds=5):
    "End-to-end FPS from dxrun (NPU and CPU parts, synthetic input)."
    lines = get_ipython().getoutput(f'dxrun -m "{model_path}" --use-ort -t {seconds}')
    match = re.search(r"FPS\s*:\s*([0-9.]+)", "\n".join(lines))
    if not match:
        print("\n".join(lines))
        raise RuntimeError(f"Could not read FPS from dxrun output for {label}.")
    print(f"{label}: {float(match.group(1)):.2f} FPS")
    return float(match.group(1))


def measure_topk(label, config, baseline_onnx, tails, baseline_dxnn, topk_dxnn, tensor):
    feeds = head_values(baseline_onnx, config, tensor)
    cpu_ms = [time_cpu_part(tails[name], feeds) for name in ("baseline", "TopK")]
    fps = [benchmark_dxnn(f"{label} baseline", baseline_dxnn), benchmark_dxnn(f"{label} TopK 300", topk_dxnn)]
    return label, cpu_ms, fps


def show_measurements(rows):
    table = [
        "| Model | CPU part, baseline (ms) | CPU part, TopK (ms) | CPU time saved | "
        "`dxrun` FPS, baseline | `dxrun` FPS, TopK | Relative throughput |",
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
    for label, (cpu_base, cpu_topk), (fps_base, fps_topk) in rows:
        table.append(f"| {label} | {cpu_base:.3f} | {cpu_topk:.3f} | {1 - cpu_topk / cpu_base:.0%} | "
                     f"{fps_base:.2f} | {fps_topk:.2f} | {fps_topk / fps_base:.2f}× |")
    display(Markdown("\n".join(table)))


yolo26_measurement = measure_topk(f"YOLO26-{YOLO26_VARIANT} det", yolo26_pass_config, YOLO26_BASELINE, yolo26_tails,
                                  yolo26_baseline_dxnn, yolo26_topk_dxnn, test_tensor)
show_measurements([yolo26_measurement])

How to read the result:

- **CPU part.** This is what the pass changes. For the Model Zoo models it shrinks by about 35 to 45 percent. The absolute time depends on the host CPU: the same work takes several times longer on an Arm core than on an x86_64 desktop core.
- **`dxrun` FPS.** It improves only as far as the CPU part limits the pipeline. On the x86_64 desktop with a DX-M1 used to test this notebook, YOLO26-n measured about 0.74 ms → 0.41 ms for the CPU part but only about 1.06× in `dxrun` FPS, because the NPU was the slower stage.
- **On an Arm host the CPU part is the limit.** The same DXNN files from this section, measured on a Raspberry Pi 5 with a DX-M1 (`dxrun -m <model> --use-ort -t 10`, default settings):

  | YOLO26-n | Baseline FPS | TopK 300 FPS | Speed-up |
  |---|---:|---:|---:|
  | det | 100.79 | 179.54 | 1.78× |
  | seg | 64.13 | 95.61 | 1.49× |
  | pose | 142.48 | 265.44 | 1.86× |

  For the three baseline models, DX-RT also printed `CPU TASK [cpu_0] Inference Worker - Average Input Queue Load : ...% (DXRT_DYNAMIC_CPU_THREAD: OFF) - To improve FPS, set: 'export DXRT_DYNAMIC_CPU_THREAD=ON'`. That message means frames were waiting for the CPU task. It did not appear for the TopK models. When you see it on your board, the CPU part is the bottleneck and this optimization is likely to help. `DXRT_DYNAMIC_CPU_THREAD=ON` is a separate setting that gives the CPU task more threads; it was not used for these numbers.
- **Reference from the DX-COM manual**, on a DX-M1 with an ARM Cortex-A53 host (`DXRT_DYNAMIC_CPU_THREAD=ON dxrun -m <model> -l 1000 --use-ort`): det 184.89 → 315.73 FPS (1.7×), seg 130.82 → 195.05 FPS (1.5×), pose 242.77 → 304.38 FPS (1.3×).

To decide for your own product, run this section on the target board itself.

### 6.8 Run the TopK DXNN in DX-APP

The application code does not change. DX-APP's YOLO26 post-processors read the first output by position, and the DX-STREAM `yolo26_od`, `yolo26_seg`, and `yolo26_pose` post-process libraries select outputs by shape (`[1, 300, 6]`, at least 38 columns, at least 57 columns). The renamed output (`output0` → `preopt_output`) therefore needs no change. Both still need ONNX Runtime for the CPU task.

The next cell runs the DX-APP Python example for the model with each DXNN on the same image and shows both results. It uses the DX-RT Python environment from Tutorial 01 and the DX-APP sources from Tutorial 02; if they are missing, the cell skips. For the default `n` variant it is equivalent to:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
<DX_ALL_SUITE_DIR>/dx-runtime/venv-dx-runtime/bin/python \
  src/python_example/object_detection/yolo26/yolo26-n_640x640/yolo26-n_640x640_sync.py \
  --model <workspace>/outputs/yolo26n_topk300/yolo26-n_640x640_topk300.dxnn \
  --image sample/img/sample_people.jpg --no-display --save --save-dir <workspace>/outputs/dx_app_results
```

On this notebook's test machine both DXNN files found the same objects; box corners differed by up to about 2 pixels and scores by up to about 0.05, because each model is quantized separately. Check accuracy on your validation set before you deploy, for example with the DX Model Zoo evaluation tools.

In [ ]:
DX_RUNTIME_PYTHON = DX_RUNTIME_DIR / "venv-dx-runtime" / "bin" / "python"   # has dx_engine
DX_APP_RESULTS = OUTPUT_DIR / "dx_app_results"
DX_APP_READY = DX_RUNTIME_PYTHON.is_file() and (DX_APP_DIR / "src" / "python_example").is_dir()
DX_APP_EXAMPLES = {
    "base": "object_detection/yolo26",
    "seg": "instance_segmentation/yolo26_seg",
    "pose": "pose_estimation/yolo26_pose",
}


def dx_app_example(task, model_name):
    "Path of the DX-APP Python example for a Model Zoo YOLO26 model."
    return DX_APP_DIR / "src" / "python_example" / DX_APP_EXAMPLES[task] / model_name / f"{model_name}_sync.py"


def run_dx_app(example, dxnn, image, tag):
    "Run a DX-APP Python example on one image and return the saved result image."
    save_dir = DX_APP_RESULTS / tag
    shutil.rmtree(save_dir, ignore_errors=True)
    log = get_ipython().getoutput(
        f'cd "{DX_APP_DIR}" && "{DX_RUNTIME_PYTHON}" "{example}" --model "{dxnn}" --image "{image}" '
        f'--no-display --save --save-dir "{save_dir}" 2>&1')
    results = sorted(save_dir.rglob("*_result.*"))
    if not results:
        print("\n".join(log[-20:]))
        raise RuntimeError(f"DX-APP did not save a result for {tag}; see its output above.")
    return cv2.imread(str(results[-1]))


if DX_APP_READY:
    example = dx_app_example("base", YOLO26_NAME)
    show_images({
        f"DX-APP, YOLO26-{YOLO26_VARIANT} baseline DXNN":
            run_dx_app(example, yolo26_baseline_dxnn, TEST_IMAGE, f"yolo26{YOLO26_VARIANT}_baseline"),
        f"DX-APP, YOLO26-{YOLO26_VARIANT} TopK DXNN":
            run_dx_app(example, yolo26_topk_dxnn, TEST_IMAGE, f"yolo26{YOLO26_VARIANT}_topk300"),
    })
else:
    print(f"Skipped: DX-APP or {DX_RUNTIME_PYTHON} was not found (Tutorials 01 and 02 install them).")

### 6.9 Segmentation and pose: the same pass with `task`

DX-COM v2.5.0 extends `yolo_no_dfl_postprocess` to YOLO26 segmentation and pose models. The pass name, `topk`, and the `bbox` and `cls_conf` tensors stay as in 6.3. Two things change: the `task` key, and one extra head tensor per scale.

| `task` | Model Zoo model (`n`) | Head tensors per scale | `num_classes` | Output after the pass |
|---|---|---|---:|---|
| `base` | `yolo26-n_640x640` | `bbox` (4 ch), `cls_conf` (80 ch) | 80 | `[1, 300, 6]` |
| `seg` | `yolo26-n-seg_640x640` | + `mask_coeff` (32 ch), `one2one_cv4.<scale>` | 80 | `[1, 300, 38]` and the prototype `[1, 32, 160, 160]` |
| `pose` | `yolo26-n-pose_640x640` | + `kpt` (51 ch), `one2one_cv4_kpts.<scale>`, with `kpt_shape` `[17, 3]` | 1 | `[1, 300, 57]` |

- **Segmentation.** Each of the 300 kept candidates carries its 32 mask coefficients. The mask prototype output is passed through unchanged; the host still combines coefficients and prototypes into masks.
- **Pose.** The keypoint head is a single Conv, so its tensor name (`/model.23/one2one_cv4_kpts.<scale>/Conv_output_0`) does not follow the `...<scale>.2/Conv_output_0` pattern of the other branches. The rules in 6.2 do not depend on names, so `yolo26_heads.py` finds it anyway. `kpt_shape` comes from the ONNX metadata, and 17 x 3 must equal the 51 channels of the keypoint Conv.
- **Validation.** `pre_optimize` checks the head tensors against `task` before it edits the graph: `seg` requires `mask_coeff`, `pose` requires `kpt` and `kpt_shape`, and each rejects the other's tensor. It cannot detect a segmentation model run with `task="base"`: that compiles, but the masks are lost. `yolo26_heads.py` therefore compares `task` with the model metadata.

The next cells repeat 6.1 to 6.8 for both models, one step per cell. They follow `YOLO26_VARIANT`, because the Model Zoo publishes `seg` and `pose` models for every size.

#### 6.9.1 Download the models and find the head tensors

In [ ]:
yolo26_tasks = {}
for task in ("seg", "pose"):
    task_name = f"yolo26-{YOLO26_VARIANT}-{task}_640x640"
    files = yolo26_tasks[task] = {
        "name": task_name,
        "baseline": MODEL_DIR / f"{task_name}.onnx",
        "topk": MODEL_DIR / f"{task_name}_topk300.onnx",
        "modelzoo_json": CONFIG_DIR / f"{task_name}.modelzoo.json",
        "local_json": CONFIG_DIR / f"{task_name}.local.json",
        "baseline_output": OUTPUT_DIR / f"yolo26{YOLO26_VARIANT}_{task}_baseline",
        "topk_output": OUTPUT_DIR / f"yolo26{YOLO26_VARIANT}_{task}_topk300",
    }
    print(f"===== YOLO26-{YOLO26_VARIANT} {task} ({task_name}) =====")
    download_file(f"https://sdk.deepx.ai/modelzoo/onnx/{task_name}.onnx", files["baseline"])
    download_file(f"https://sdk.deepx.ai/modelzoo/q-lite-json/2_5_0/{task_name}.json", files["modelzoo_json"])

    task_model = onnx.load(files["baseline"])
    files["names"] = read_metadata(task_model)["names"]
    files["config"] = find_yolo26_heads(task_model, task=task)
    print(f"task={task}, num_classes={files['config']['num_classes']}, "
          f"kpt_shape={files['config']['layers'][0].get('kpt_shape')}")
    print("\n".join(describe_heads(task_model, files["config"])) + "\n")

#### 6.9.2 Apply the pass and compare the CPU part

For the default `n` variant the optimize step is equivalent to:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/notebooks/T05-DX-Compiler/workspace
python optimize_yolo26.py models/yolo26-n-seg_640x640.onnx models/yolo26-n-seg_640x640_topk300.onnx --task seg
python optimize_yolo26.py models/yolo26-n-pose_640x640.onnx models/yolo26-n-pose_640x640_topk300.onnx --task pose
```

`--task` is optional here, because the script reads it from the metadata. Passing it states what you expect, and the script stops if the model disagrees. In the pose table, look at the keypoint `Sigmoid` as well: it moves from `[1, 17, 1, 8400]` to `[1, 300, 17, 1]`.

In [ ]:
for task, files in yolo26_tasks.items():
    baseline_onnx, topk_onnx = files["baseline"], files["topk"]
    !"{DXCOM_PYTHON}" "{TOPK_SCRIPT}" "{baseline_onnx}" "{topk_onnx}" --task {task} | tail -n 1
    files["tails"] = compare_cpu_tails(baseline_onnx, topk_onnx, files["config"], f"YOLO26-{YOLO26_VARIANT} {task}")

#### 6.9.3 Check that the results are identical

The comparison covers the mask coefficients and the prototype for segmentation, and the keypoints for pose. The images draw the masks and keypoints from each model's own output.

In [ ]:
for task, files in yolo26_tasks.items():
    print(f"===== YOLO26-{YOLO26_VARIANT} {task} =====")
    baseline_out, topk_out = compare_onnx_outputs(files["baseline"], files["topk"], test_tensor)
    kpt_shape = files["config"]["layers"][0].get("kpt_shape")
    show_images({
        f"{task} baseline ONNX": draw_results(test_canvas, baseline_out, files["names"], task, kpt_shape),
        f"{task} TopK ONNX": draw_results(test_canvas, topk_out, files["names"], task, kpt_shape),
    })

#### 6.9.4 Compile the baseline and TopK models

The cell writes the local JSON for each model (as in 6.6.1) and compiles four models with the same options as 6.6. For the default `n` variant it is equivalent to the following commands (replace `seg` with `pose` for the pose models):

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/notebooks/T05-DX-Compiler/workspace
dxcom -m models/yolo26-n-seg_640x640.onnx \
      -c configs/yolo26-n-seg_640x640.local.json \
      -o outputs/yolo26n_seg_baseline \
      --gen_log \
      --export_html
dxcom -m models/yolo26-n-seg_640x640_topk300.onnx \
      -c configs/yolo26-n-seg_640x640.local.json \
      -o outputs/yolo26n_seg_topk300 \
      --gen_log \
      --export_html
```

If an output directory already contains a DXNN file, the cell reports that file and skips `dxcom` for that model.

In [ ]:
def compile_once(onnx_path, config_path, output_dir):
    "Compile with dxcom unless output_dir already holds a DXNN file; return the DXNN path."
    dxnn_files = list(output_dir.glob("*.dxnn"))
    if dxnn_files:
        print(f"Skip compilation: found {dxnn_files[0]}")
    else:
        print(f"Compiling {onnx_path.name} with {config_path.name} -> {output_dir}")
        !source "{DX_COMPILER_VENV}/bin/activate" && \
          dxcom -m "{onnx_path}" \
                -c "{config_path}" \
                -o "{output_dir}" \
                --gen_log \
                --export_html \
                2>&1 | sed -u 's/\x1b\[A//g; s/\r/\n/g' | grep --line-buffered -vP '^\s*$|\d+(\.\d+)?/\d+(\.\d+)? \['
        dxnn_files = list(output_dir.glob("*.dxnn"))
    if not dxnn_files:
        raise FileNotFoundError(f"Compiling {onnx_path.name} did not produce a DXNN file.")
    return max(dxnn_files, key=lambda path: path.stat().st_mtime_ns)


for task, files in yolo26_tasks.items():
    task_config = json.loads(files["modelzoo_json"].read_text())
    task_config["default_loader"]["dataset_path"] = "./calibration_dataset"
    files["local_json"].write_text(json.dumps(task_config, indent=2) + "\n")
    files["baseline_dxnn"] = compile_once(files["baseline"], files["local_json"], files["baseline_output"])
    files["topk_dxnn"] = compile_once(files["topk"], files["local_json"], files["topk_output"])

#### 6.9.5 Measure detection, segmentation, and pose

The cell measures the CPU part and `dxrun` FPS of the four new DXNN files as in 6.7, and adds the detection result from 6.7 to the table.

In [ ]:
yolo26_task_measurements = [yolo26_measurement]
for task, files in yolo26_tasks.items():
    yolo26_task_measurements.append(measure_topk(
        f"YOLO26-{YOLO26_VARIANT} {task}", files["config"], files["baseline"], files["tails"],
        files["baseline_dxnn"], files["topk_dxnn"], test_tensor))
show_measurements(yolo26_task_measurements)

How the three tasks compare:

- **Detection and segmentation** save CPU time in the same place: the class-score `Sigmoid` over 80 classes x 8,400 candidates. Segmentation also keeps the mask prototype output, which the pass does not change, and the host still builds the masks, so its relative gain is smaller.
- **Pose** has a single class, so its class-score work is small. Its saving comes from the keypoint visibility `Sigmoid` (17 x 8,400 values) and the keypoint decoding, which now run for 300 candidates only. How much that matters depends on the host: on the Raspberry Pi 5 above, pose gained the most (1.86×), while in the manual's Cortex-A53 results it gained the least (1.3×).

#### 6.9.6 Run the segmentation and pose DXNN files in DX-APP

In [ ]:
if DX_APP_READY:
    for task, files in yolo26_tasks.items():
        example = dx_app_example(task, files["name"])
        show_images({
            f"DX-APP, {task} baseline DXNN": run_dx_app(example, files["baseline_dxnn"], TEST_IMAGE, f"{task}_baseline"),
            f"DX-APP, {task} TopK DXNN": run_dx_app(example, files["topk_dxnn"], TEST_IMAGE, f"{task}_topk300"),
        })
else:
    print("Skipped: DX-APP is not available (see 6.8).")

## 7. Troubleshooting

| Symptom | What you see | Likely cause | Fix |
|---|---|---|---|
| A PPU node name is not found | `ValueError: ... not found in the ONNX graph` | The `layer` names in the PPU configuration belong to another export | Inspect the ONNX in Netron and update the node names in the JSON |
| `dxrun` finds no device | `No device found` or `Fail to initialize device` | No DEEPX NPU on this host | Run the comparison on a host with an NPU; compile-only steps still work |
| FPS cannot be read | `Could not read FPS from dxrun output` | The `dxrun` output format changed | Print the full output above and read the FPS line manually |
| A JSON change has no effect | `Skip compilation: found ...` | The output directory already contains a DXNN | Delete `workspace/outputs/<dir>` and rerun |
| `pre_optimize` rejects the head tensors | `ValueError: [yolo_no_dfl_postprocess] task='seg' requires 'mask_coeff' on every layer` or `Cannot find shape for mask_coeff=...` | `task` and the head tensors do not match the model | Find the tensors with `find_yolo26_heads()` on this exact ONNX file |
| The head finder stops at the task check | `The model metadata says task='seg', not 'base'` | The task argument does not match the model | Pass the task from the message; with `base`, a segmentation model would compile without its masks |
| The head finder reports a one-to-many head | `This ONNX file has the one-to-many head (output [1, 4 + classes, 8400])` | The model was exported without `nms=False` | Re-export with `yolo export ... format=onnx nms=False` |
| The head finder finds no `cls_conf` Conv | `expected one cls_conf head Conv with N channels, found none` | The class count is wrong or missing from the metadata | Pass `num_classes` (`--num-classes` for the script) |
| The head finder reports a dynamic input | `The input height is dynamic` | The model was exported with `dynamic=True` | Re-export with a fixed `imgsz` |
| The TopK DXNN is not faster in `dxrun` | Relative throughput close to 1.00× | The NPU, not the host CPU, limits the pipeline (common on x86_64), or the model has few classes | Compare the CPU-part times in 6.7, and measure on the target board |
| `pre_optimize` in the JSON has no effect | `falling back to legacy: ... 'pre_optimize' and 'ppu' are mutually exclusive` | The JSON also has a `ppu` block | Remove `ppu` from the JSON |
| DX-APP shows COCO labels for a model with other classes | COCO class names on your objects | The labels come from the example's own `config.json`; `--config` does not change them | Export a standalone package with `scripts/extract_model_package.sh --no-prune` (Tutorial 02) and set `num_classes` and `visualizer.kwargs.custom_labels` in its `config.json` |
| A download is incomplete | A tiny or zero-byte file in `workspace/models` | The download was interrupted | Delete the file; the download helper verifies size before use |


## 8. Summary

### 8.1 Intermediate optimization map

```text
Representative calibration data
              │
              ▼
     Inspect the detection head
              │
      ┌───────┴────────┐
      │                │
Anchor-based      Anchor-free
  YOLOv7             YOLOX
      │                │
PPU Type 0        PPU Type 1
      │                │
      └───────┬────────┘
              │
              ▼
 Reduce host-side filtering
 and class-selection workload

YOLO26 det / seg / pose head
          │
          ▼
 Find head tensors → TopK-first ONNX rewrite
 (task = base / seg / pose)
          │
          ▼
 Same results, less host-CPU work
 per frame
```

### 8.2 Three optimization paths

| Topic | Model used | Main decision | Optimization location | Host CPU responsibility |
|---|---|---|---|---|
| PPU Type 0 | YOLOv7 | Select an anchor-based head and map its Conv nodes | Hardware PPU | NMS and application logic |
| PPU Type 1 | YOLOX-S | Map bbox, objectness, and class branches | Hardware PPU | NMS and application logic |
| TopK-first optimization | YOLO26 det, seg, and pose (`YOLO26_VARIANT`, default n) | Move TopK before Sigmoid and decoding; set `task` and the head tensors | Rewritten ONNX graph (Python API or compile JSON) | Decode fewer candidates (with their mask coefficients or keypoints) |

### 8.3 Experiments completed

| Experiment | Compared models | Controlled variable | Verification |
|---|---|---|---|
| Type 0 PPU | YOLOv7 non-PPU vs PPU | Hardware post-processing | `dxparse`, `dxrun` |
| Type 1 PPU | YOLOX-S non-PPU vs PPU | Hardware post-processing | `dxparse`, `dxrun` |
| TopK optimization | YOLO26 det, seg, pose: baseline vs TopK 300 (selected size) | ONNX graph structure | ONNX Runtime output comparison, CPU-part time, `dxparse`, `dxrun`, DX-APP |

### 8.4 Completion checklist

- [ ] Treat calibration data and preprocessing as part of the model definition
- [ ] Identify the boundary between NPU, PPU, and host CPU processing
- [ ] Verify the YOLOv7 Type 0 Conv-node mapping
- [ ] Compile and inspect a Type 0 PPU model
- [ ] Verify the YOLOX-S Type 1 bbox, objectness, and class mappings
- [ ] Compile and inspect a Type 1 PPU model
- [ ] Compare PPU and non-PPU models under the same benchmark conditions
- [ ] Explain which part of a YOLO26 DXNN runs on the host CPU
- [ ] Find YOLO26 head tensors with `find_yolo26_heads()` instead of copying names
- [ ] Apply TopK-first optimization to a YOLO26 model (and try a second size)
- [ ] Confirm with ONNX Runtime that the baseline and TopK models give identical outputs
- [ ] Measure the CPU-part time and `dxrun` FPS, and explain why they differ
- [ ] Run the TopK DXNN in DX-APP without changing the application
- [ ] Apply the same pass to YOLO26 segmentation and pose models with `task` = `seg` and `pose`
- [ ] Validate accuracy, host CPU usage, and full-pipeline latency with real data

### 8.5 How to choose

| If your model has... | Start with... |
|---|---|
| An anchor-based YOLO head | PPU Type 0 |
| Separate bbox, objectness, and class branches | PPU Type 1 |
| A large candidate set followed by TopK, on a host where the CPU part limits FPS | TopK-first graph optimization |
| A segmentation or pose head (YOLO26 or YOLOv8 to v13) | TopK-first graph optimization with `task` = `seg` or `pose` |
| An unknown or customized head | Inspect the exact ONNX graph before selecting an optimization |

> **Remember:** lower host workload and higher synthetic-input FPS are not the same result. Use `dxrun` for a controlled runtime comparison, then measure task accuracy, host CPU usage, and end-to-end latency with the real application workload.

### 8.6 Next step

Continue with the **Advanced tutorial** to learn:

- Q-Lite, Q-PRO, and Q-Master selection;
- quantization diagnosis;
- QXNN resume workflows;
- QAT;
- Python API compilation; and
- advanced compiler controls.